# 07-3. JSON API와 응답 검증 예제

## Goal

- JSON 문법 오류와 계약 오류를 구분합니다.
- 중복 키를 거부합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

합성 JSON 문자열만 파싱합니다.


## Steps

### 엄격한 JSON 객체 검증

파싱 성공만으로 필요한 필드와 자료형이 맞다고 판단하지 않습니다.


In [1]:
import json


def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키: {key}")
        result[key] = value
    return result


def parse_health(payload: bytes) -> dict:
    data = json.loads(payload.decode("utf-8"), object_pairs_hook=reject_duplicate_keys)
    if not isinstance(data, dict):
        raise ValueError("최상위 값은 객체여야 합니다")
    if data.get("status") != "ok" or not isinstance(data.get("service"), str):
        raise ValueError("응답 계약이 맞지 않습니다")
    return data


health = parse_health(b'{"status":"ok","service":"training"}')
print(health)


{'status': 'ok', 'service': 'training'}


## Checks

계약 오류와 중복 키를 각각 거부합니다.


In [2]:
for payload in (b'{"status":"down"}', b'{"status":"ok","status":"down","service":"x"}'):
    try:
        parse_health(payload)
    except ValueError as error:
        print("예상한 오류:", error)
    else:
        raise AssertionError("잘못된 JSON 계약을 허용했습니다")


예상한 오류: 응답 계약이 맞지 않습니다
예상한 오류: 중복 JSON 키: status


## Next Steps

오류 응답도 JSON일 수 있으므로 상태 코드와 오류 스키마를 별도로 정의합니다.
